In [ ]:
# ==========================================
# Stock Price Prediction using LSTM
# ==========================================

# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import math
import yfinance as yf

# ==========================================
# Step 1: Download Stock Data
# ==========================================

stock_symbol = 'AAPL'  # Apple Stock
start_date = '2015-01-01'
end_date = '2025-01-01'

data = yf.download(stock_symbol, start=start_date, end=end_date)

# Display first 5 rows
print(data.head())

# ==========================================
# Step 2: Visualize Closing Price
# ==========================================

plt.figure(figsize=(14, 6))
plt.plot(data['Close'])
plt.title(f'{stock_symbol} Closing Price')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.show()

# ==========================================
# Step 3: Prepare Dataset
# ==========================================

close_data = data['Close'].values
close_data = close_data.reshape(-1, 1)

# Normalize data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(close_data)

# Training data size
training_size = int(len(scaled_data) * 0.80)

train_data = scaled_data[:training_size]
test_data = scaled_data[training_size:]

# ==========================================
# Step 4: Create Dataset Function
# ==========================================

def create_dataset(dataset, time_step=60):
    X, y = [], []

    for i in range(time_step, len(dataset)):
        X.append(dataset[i-time_step:i, 0])
        y.append(dataset[i, 0])

    return np.array(X), np.array(y)

time_step = 60

X_train, y_train = create_dataset(train_data, time_step)
X_test, y_test = create_dataset(test_data, time_step)

# Reshape for LSTM
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

# ==========================================
# Step 5: Build LSTM Model
# ==========================================

model = Sequential()

# First LSTM Layer
model.add(LSTM(units=50, return_sequences=True,
               input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))

# Second LSTM Layer
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

# Third LSTM Layer
model.add(LSTM(units=50))
model.add(Dropout(0.2))

# Output Layer
model.add(Dense(units=1))

# Compile Model
model.compile(optimizer='adam', loss='mean_squared_error')

# Model Summary
model.summary()

# ==========================================
# Step 6: Train the Model
# ==========================================

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    verbose=1
)

# ==========================================
# Step 7: Predict Stock Prices
# ==========================================

train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Inverse Transform
train_predict = scaler.inverse_transform(train_predict)
test_predict = scaler.inverse_transform(test_predict)

y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# ==========================================
# Step 8: Calculate RMSE
# ==========================================

train_rmse = math.sqrt(
    mean_squared_error(y_train_actual, train_predict)
)

test_rmse = math.sqrt(
    mean_squared_error(y_test_actual, test_predict)
)

print(f"Train RMSE: {train_rmse}")
print(f"Test RMSE: {test_rmse}")

# ==========================================
# Step 9: Visualization
# ==========================================

# Prepare plotting arrays
train_plot = np.empty_like(scaled_data)
train_plot[:, :] = np.nan
train_plot[time_step:len(train_predict)+time_step, :] = train_predict

test_plot = np.empty_like(scaled_data)
test_plot[:, :] = np.nan
test_plot[len(train_predict)+(time_step*2):len(scaled_data), :] = test_predict

# Plot
plt.figure(figsize=(16, 8))

plt.plot(
    scaler.inverse_transform(scaled_data),
    label='Actual Stock Price'
)

plt.plot(train_plot, label='Training Prediction')

plt.plot(test_plot, label='Testing Prediction')

plt.title(f'{stock_symbol} Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.grid(True)
plt.show()

# ==========================================
# Step 10: Predict Future Price
# ==========================================

future_days = 30

last_60_days = scaled_data[-60:]
future_input = last_60_days.reshape(1, 60, 1)

future_predictions = []

for i in range(future_days):

    pred = model.predict(future_input, verbose=0)

    future_predictions.append(pred[0][0])

    # Update input sequence
    future_input = np.append(
        future_input[:, 1:, :],
        [[[pred[0][0]]]],
        axis=1
    )

# Convert back to original scale
future_predictions = scaler.inverse_transform(
    np.array(future_predictions).reshape(-1, 1)
)

# ==========================================
# Step 11: Plot Future Predictions
# ==========================================

plt.figure(figsize=(12, 6))

plt.plot(
    future_predictions,
    marker='o'
)

plt.title(f'Next {future_days} Days Prediction')
plt.xlabel('Days')
plt.ylabel('Predicted Price')
plt.grid(True)
plt.show()

print("Future Predictions:")
print(future_predictions)